# Monte Carlo Simulation - Companion Notebook (parts 3 and 4)

**The second half of the pair.** Every piece of code from parts 3 and 4 of the
[Monte Carlo guide](https://statexampro.com/lessons/monte-carlo/) on Stat Exam Pro - and every number quoted in those two lessons is
this notebook's actual output. Seeds are fixed, so you will reproduce them to the digit.

Assumes parts 1 and 2: the square-root law, standard errors, seeds, the inverse transform, the five-step recipe. If
those aren't reflexes yet, start with the [entry notebook](https://colab.research.google.com/github/StatExamPro/statexampro.com/blob/main/notebooks/monte-carlo-entry.ipynb).

| The lesson | The code |
|---|---|
| [Part 3 - Paths, options, and correlated inputs](https://statexampro.com/lessons/monte-carlo/paths-and-options/) | sections 1-3 below: path simulators and their certification tests, Black-Scholes, Cholesky, copulas |
| [Part 4 - Beating the square-root law](https://statexampro.com/lessons/monte-carlo/variance-reduction/) | sections 4-8 below: the variance-reduction scorecard, importance sampling, Sobol, cluster trials, Metropolis |

Requirements: `numpy`, `scipy` (>=1.7 for `scipy.stats.qmc`), `matplotlib` - all preinstalled in Colab. Runtime: ~2-3 minutes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, qmc

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
rng = np.random.default_rng(314)
print(np.__version__)

## Section 1 — Simulating Processes in Time: Geometric Brownian Motion

GBM: dS = μS dt + σS dW. Its exact solution — **log-returns are normal**:
S(t+Δ) = S(t) · exp( (μ − σ²/2)Δ + σ√Δ·Z ). We simulate paths by cumulatively summing normal *log*-increments, then exponentiating. Rows = worlds, columns = time (Part I habit).
> Explained in the guide: [Part 3 - what changes when time enters](https://statexampro.com/lessons/monte-carlo/paths-and-options/#paths)


In [ ]:
def gbm_paths(S0, mu, sigma, T, steps, N, rng):
    dt = T / steps
    z = rng.standard_normal((N, steps))
    log_incr = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
    return S0 * np.exp(np.cumsum(log_incr, axis=1))     # shape (N, steps)

S0, mu, sigma, T = 100.0, 0.08, 0.25, 1.0
paths = gbm_paths(S0, mu, sigma, T, steps=252, N=100_000, rng=rng)

t = np.linspace(1/252, T, 252)
plt.plot(t, paths[:30].T, lw=0.7, alpha=0.8)
plt.plot(t, S0 * np.exp(mu * t), "k--", lw=2, label=r"$\mathbb{E}[S_t] = S_0 e^{\mu t}$")
plt.xlabel("years"); plt.ylabel("price"); plt.legend()
plt.title("30 GBM paths (of 100,000)"); plt.show()

print(f"E[S_T]: simulated {paths[:, -1].mean():.2f}  theory {S0*np.exp(mu*T):.2f}")
print(f"terminal price median {np.median(paths[:,-1]):.2f} < mean — lognormal skew")

**The −σ²/2 bug, live.** The single most common GBM bug: writing `exp(μΔ + σ√Δ Z)` without the −σ²/2 correction. Each step then has expected growth e^{μΔ+σ²Δ/2} > e^{μΔ} (lognormal mean formula), and the bias **compounds** — your simulated asset silently drifts upward. With σ = 25% that's ≈ 3.2% of free fake return per year.

In [ ]:
z = np.random.default_rng(1).standard_normal((200_000, 252))
dt = 1/252
buggy   = S0 * np.exp(np.cumsum(mu*dt + sigma*np.sqrt(dt)*z, axis=1))          # WRONG
correct = S0 * np.exp(np.cumsum((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*z, axis=1))

print(f"theory     E[S_T] = {S0*np.exp(mu*T):7.2f}")
print(f"correct MC E[S_T] = {correct[:,-1].mean():7.2f}")
print(f"buggy   MC E[S_T] = {buggy[:,-1].mean():7.2f}   <- extra e^(σ²T/2) = "
      f"{np.exp(0.5*sigma**2*T):.4f}x of phantom drift")

**Exact vs Euler discretization.** GBM is a luxury: the exact one-step formula above has *zero* discretization error at any Δ. The generic Euler scheme S(t+Δ)=S(t)(1+μΔ+σ√Δ Z) is what you fall back on for processes without closed-form solutions (most of them) — it's biased at finite Δ, with bias shrinking as Δ→0. Know which regime you're in.

In [ ]:
r0 = np.random.default_rng(2)
z1 = r0.standard_normal(2_000_000)
exact_ST = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z1)   # one exact step for S_T!
print(f"one exact step, E[S_T] = {exact_ST.mean():.3f} (theory {S0*np.exp(mu*T):.3f})")

for steps in [1, 4, 12, 252]:
    zz = np.random.default_rng(3).standard_normal((500_000, steps))
    dt = T / steps
    s = S0 * np.prod(1 + mu*dt + sigma*np.sqrt(dt)*zz, axis=1)        # Euler
    print(f"Euler, {steps:3d} steps: E[S_T] = {s.mean():7.3f}")

## Section 2 — Option Pricing, Validated Against Black–Scholes

Risk-neutral pricing recipe: simulate GBM **with drift = r** (not μ!), average the payoff, discount at r. First the European call — the great solvable case — as our certification target.
> Explained in the guide: [Part 3 - option pricing, certified against Black-Scholes](https://statexampro.com/lessons/monte-carlo/paths-and-options/#pricing)


In [ ]:
def bs_call(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.25, 1.0
N = 1_000_000
rng2 = np.random.default_rng(7)
Z = rng2.standard_normal(N)
ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)     # risk-neutral drift!
disc_payoff = np.exp(-r*T) * np.maximum(ST - K, 0.0)

mc, se = disc_payoff.mean(), disc_payoff.std(ddof=1)/np.sqrt(N)
bs = bs_call(S0, K, r, sigma, T)
print(f"Black–Scholes : {bs:.4f}")
print(f"Monte Carlo   : {mc:.4f} ± {1.96*se:.4f}   (|error|/se = {abs(mc-bs)/se:.2f})")

**Greeks and common random numbers.** Delta = ∂price/∂S₀ by finite differences. Bump S₀ by h with *independent* draws and the difference of two noisy numbers is mostly noise; with **common random numbers** (same Z for both bumps) the noise cancels in the subtraction. Same trick, spectacularly different standard errors. Bonus: the *pathwise* estimator — differentiate the payoff along each path — needs no bump at all.

In [ ]:
h = 1.0
rngA, rngB = np.random.default_rng(10), np.random.default_rng(11)

# independent draws for the two bumps (DON'T)
Zu, Zd = rngA.standard_normal(N), rngB.standard_normal(N)
up  = np.exp(-r*T)*np.maximum((S0+h)*np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*Zu) - K, 0)
dn  = np.exp(-r*T)*np.maximum((S0-h)*np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*Zd) - K, 0)
delta_indep = (up - dn) / (2*h)

# common random numbers (DO)
Zc = np.random.default_rng(12).standard_normal(N)
STc = np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*Zc)
up  = np.exp(-r*T)*np.maximum((S0+h)*STc - K, 0)
dn  = np.exp(-r*T)*np.maximum((S0-h)*STc - K, 0)
delta_crn = (up - dn) / (2*h)

# pathwise estimator: d payoff/d S0 = e^{-rT} 1[S_T>K] * S_T/S0
delta_pw = np.exp(-r*T) * (STc*S0 > K) * STc

d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
print(f"true delta (BS)          : {norm.cdf(d1):.4f}")
for name, d in [("independent bumps", delta_indep), ("common random numbers", delta_crn),
                ("pathwise", delta_pw)]:
    print(f"{name:22s}: {d.mean():.4f} ± {1.96*d.std(ddof=1)/np.sqrt(N):.4f}")
print(f"\nvariance ratio indep/CRN = {delta_indep.var()/delta_crn.var():,.0f}x")

**Where MC earns its salary: the arithmetic Asian option.** Payoff on the *average* price over 52 weekly observations — path-dependent, no closed form, a 52-dimensional integral. Grids are dead; MC doesn't notice.

In [ ]:
def asian_mc(N, rng, steps=52, return_paths=False):
    dt = T / steps
    z = rng.standard_normal((N, steps))
    S = S0 * np.exp(np.cumsum((r - .5*sigma**2)*dt + sigma*np.sqrt(dt)*z, axis=1))
    A = S.mean(axis=1)                                   # arithmetic average price
    Y = np.exp(-r*T) * np.maximum(A - K, 0.0)            # Asian call payoffs
    G = np.exp(np.log(S).mean(axis=1))                   # geometric average (for Sec. 4)
    X = np.exp(-r*T) * np.maximum(G - K, 0.0)
    return (Y, X, S) if return_paths else (Y, X)

Y, X = asian_mc(200_000, np.random.default_rng(20))
print(f"arithmetic Asian call: {Y.mean():.4f} ± {1.96*Y.std(ddof=1)/np.sqrt(len(Y)):.4f}")

## Section 3 — Correlated Inputs: Cholesky and Copulas

**Cholesky in three lines.** Want normals with correlation matrix C: factor C = LLᵀ (lower-triangular L), then X = Z Lᵀ for iid Z. Check: Cov(X) = L·Cov(Z)·Lᵀ = LLᵀ = C. Intuition: row k of L mixes k independent noise sources with weights chosen to hit exactly the required covariances with all previous variables.
> Explained in the guide: [Part 3 - Cholesky, copulas, and the 2008 lesson](https://statexampro.com/lessons/monte-carlo/paths-and-options/#cholesky)


In [ ]:
C = np.array([[1.0, 0.6, 0.3],
              [0.6, 1.0, 0.5],
              [0.3, 0.5, 1.0]])
L = np.linalg.cholesky(C)
Z = np.random.default_rng(30).standard_normal((500_000, 3))
Xc = Z @ L.T
print("target correlation:\n", C)
print("achieved:\n", np.corrcoef(Xc.T).round(3))

**Why correlation is a tail phenomenon.** Equal-weight portfolio of 10 assets, each Normal(0, 20%). Independent → diversification shrinks portfolio σ by √10. Correlated at ρ = 0.6 → diversification largely dies, and the loss tail explodes. Ignoring a real correlation understates risk *specifically where it hurts*.

In [ ]:
def portfolio_p5(rho, N=400_000, d=10, seed=31):
    C = np.full((d, d), rho); np.fill_diagonal(C, 1.0)
    L = np.linalg.cholesky(C)
    ret = 0.20 * (np.random.default_rng(seed).standard_normal((N, d)) @ L.T)
    port = ret.mean(axis=1)
    return port, np.percentile(port, 5)

for rho in [0.0, 0.3, 0.6, 0.9]:
    port, p5 = portfolio_p5(rho)
    print(f"rho = {rho:.1f}:  portfolio σ = {port.std():.4f}   P5 loss = {p5:+.4f}")
print(f"\ntheory, rho=0:  σ = 0.20/√10 = {0.20/np.sqrt(10):.4f}")

**Copulas: correlation beyond the normal world.** Recipe (Gaussian copula): generate correlated normals → squash each through Φ into correlated *uniforms* → push uniforms through any inverse CDFs you like. The copula carries the dependence; the marginals carry the shapes — fully decoupled.

The catch that made history: the **Gaussian copula has zero tail dependence** — extremes are asymptotically independent, joint crashes are 'impossible'. A **t-copula** (heavy-tailed mixing) makes extremes cluster. Same marginals, same overall correlation, radically different joint-crash probability — the exact modeling error at the heart of pre-2008 CDO pricing.

In [ ]:
d, rho, N = 2, 0.6, 300_000
C2 = np.array([[1, rho], [rho, 1]]); L2 = np.linalg.cholesky(C2)
rg = np.random.default_rng(33)

# Gaussian copula
U_g = norm.cdf(rg.standard_normal((N, 2)) @ L2.T)
# t-copula, df=3: correlated normals / sqrt(chi2/df) -> t -> uniforms
zz = rg.standard_normal((N, 2)) @ L2.T
w  = np.sqrt(rg.chisquare(3, size=(N, 1)) / 3)
U_t = stats.t.cdf(zz / w, df=3)

# identical lognormal marginals for both
lognorm_ppf = lambda u: np.exp(0.10 * norm.ppf(u))
Xg, Xt = lognorm_ppf(U_g), lognorm_ppf(U_t)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, X_, name in [(axes[0], Xg, "Gaussian copula"), (axes[1], Xt, "t-copula (df=3)")]:
    ax.plot(X_[:20000, 0], X_[:20000, 1], ".", ms=1, alpha=0.3)
    ax.set_title(name); ax.set_xlabel("asset 1"); ax.set_ylabel("asset 2")
fig.suptitle("Same marginals, same ρ — different tails"); fig.tight_layout(); plt.show()

for X_, name in [(Xg, "Gaussian"), (Xt, "t-copula")]:
    q = np.quantile(X_, 0.05, axis=0)
    joint = ((X_[:, 0] < q[0]) & (X_[:, 1] < q[1])).mean()
    print(f"{name:9s}: P(both assets in their worst 5%) = {joint:.3%}"
          f"   (independence would give 0.250%)")

## Section 4 — Variance Reduction I: Antithetic, Control Variates, Stratification

Error = σ/√N. Part I bought N; this section shrinks σ. Scorecard for every technique: **variance-reduction factor** = Var(naive)/Var(technique) at equal N — i.e., how many times more naive samples would buy the same accuracy.

**Antithetic variates.** Simulate in pairs (Z, −Z), average each pair. Var of a pair-average = (σ²/2)(1+ρ) where ρ = corr(f(Z), f(−Z)); monotone payoffs make ρ < 0 → cancellation better than independence. Free to implement; modest but real gains on monotone payoffs; can *backfire* (ρ>0) on symmetric/non-monotone ones.
> Explained in the guide: [Part 4 - the scorecard, antithetic, control variates](https://statexampro.com/lessons/monte-carlo/variance-reduction/#control)


In [ ]:
def euro_payoff(Z):
    ST = S0 * np.exp((r - .5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    return np.exp(-r*T) * np.maximum(ST - K, 0.0)

Npairs = 500_000
Z = np.random.default_rng(40).standard_normal(Npairs)
naive = euro_payoff(np.random.default_rng(41).standard_normal(2*Npairs))  # 2N naive draws
anti  = 0.5 * (euro_payoff(Z) + euro_payoff(-Z))                          # N pairs = 2N draws

se_naive = naive.std(ddof=1)/np.sqrt(len(naive))
se_anti  = anti.std(ddof=1)/np.sqrt(len(anti))
print(f"naive     : {naive.mean():.4f} ± {1.96*se_naive:.4f}")
print(f"antithetic: {anti.mean():.4f} ± {1.96*se_anti:.4f}")
print(f"corr(f(Z), f(-Z)) = {np.corrcoef(euro_payoff(Z), euro_payoff(-Z))[0,1]:+.3f}")
print(f"variance-reduction factor (equal budget) = {se_naive**2/se_anti**2:.1f}x")

**Control variates — the professional's favorite.** To estimate E[Y], find a sidekick X, correlated with Y, whose mean you know *exactly*. Report mean(Y) − β·(mean(X) − E[X]): the sidekick's observed error corrects Y's error. Optimal β = Cov(Y,X)/Var(X); variance shrinks by factor 1/(1−ρ²(Y,X)).

The classic: price the **arithmetic** Asian (no formula) using the **geometric** Asian (has a closed form, and ρ ≈ 0.999 with its arithmetic sibling) as control. First we *certify the control's formula by MC* — never trust a formula you haven't tested — then deploy it.

In [ ]:
def geometric_asian_closed_form(S0, K, r, sigma, T, m):
    # discrete geometric-average Asian call, averaging at t_i = iT/m
    dt = T / m
    mu_G  = np.log(S0) + (r - 0.5*sigma**2) * (T + dt) / 2
    var_G = sigma**2 * dt * (m + 1) * (2*m + 1) / (6*m)
    d1 = (mu_G - np.log(K) + var_G) / np.sqrt(var_G)
    d2 = d1 - np.sqrt(var_G)
    return np.exp(-r*T) * (np.exp(mu_G + var_G/2) * norm.cdf(d1) - K * norm.cdf(d2))

m = 52
EX = geometric_asian_closed_form(S0, K, r, sigma, T, m)
Y, X = asian_mc(400_000, np.random.default_rng(42), steps=m)
print(f"certify control: formula {EX:.4f}  vs MC {X.mean():.4f} ± {1.96*X.std(ddof=1)/np.sqrt(len(X)):.4f}")

beta = np.cov(Y, X)[0, 1] / X.var(ddof=1)
Y_cv = Y - beta * (X - EX)
se_naive = Y.std(ddof=1)/np.sqrt(len(Y))
se_cv    = Y_cv.std(ddof=1)/np.sqrt(len(Y_cv))
print(f"\ncorr(arithmetic, geometric) = {np.corrcoef(Y, X)[0,1]:.5f},  beta = {beta:.3f}")
print(f"naive           : {Y.mean():.5f} ± {1.96*se_naive:.5f}")
print(f"control variate : {Y_cv.mean():.5f} ± {1.96*se_cv:.5f}")
print(f"variance-reduction factor = {se_naive**2/se_cv**2:,.0f}x  "
      f"(≈ {se_naive**2/se_cv**2:,.0f}x fewer samples for equal accuracy)")

**Stratification in one demo.** Force the samples to cover the space evenly: split [0,1] into k strata, draw equally from each — no clumps, no gaps. Part I's π estimator, stratified on x, for a taste of the idea that QMC (Section 6) will industrialize.

In [ ]:
Ntot, k = 100_000, 100
r0 = np.random.default_rng(43)
# naive
u = r0.uniform(size=(Ntot, 2))
pi_naive = 4*((u**2).sum(1) <= 1).mean()
v_naive  = 16*((u**2).sum(1) <= 1).var()/Ntot
# stratified on x: equal draws from each vertical slice
per = Ntot // k
xs = (np.repeat(np.arange(k), per) + r0.uniform(size=Ntot)) / k
ys = r0.uniform(size=Ntot)
hits = (xs**2 + ys**2 <= 1).reshape(k, per)
pi_strat = 4*hits.mean()
v_strat  = 16*hits.var(axis=1, ddof=1).mean()/Ntot   # only WITHIN-stratum noise survives
print(f"naive      π = {pi_naive:.5f}")
print(f"stratified π = {pi_strat:.5f}")
print(f"variance-reduction factor ≈ {v_naive/v_strat:.1f}x")

## Section 5 — Importance Sampling: Hunting Rare Events

Estimate p = P(Z > 4.5) ≈ 3.4 × 10⁻⁶. Naive MC with a million draws sees ~3 events — the estimate is all noise. IS: sample where the action is (shift the distribution to N(4.5, 1)), then *reweight* each sample by the likelihood ratio w(x) = φ(x)/φ(x−4.5) = exp(−4.5x + 4.5²/2) to stay exactly unbiased. Every sample now lands near the region of interest and carries a tiny honest weight.
> Explained in the guide: [Part 4 - importance sampling: hunting rare events](https://statexampro.com/lessons/monte-carlo/variance-reduction/#is)


In [ ]:
theta, N5 = 4.5, 1_000_000
true_p = norm.sf(theta)
r5 = np.random.default_rng(50)

x_naive = r5.standard_normal(N5)
p_naive = (x_naive > theta).mean()
se_naive = np.sqrt(p_naive*(1-p_naive)/N5) if p_naive > 0 else np.nan

x_is = r5.normal(theta, 1.0, size=N5)                     # sample from the shifted law
w    = np.exp(-theta*x_is + theta**2/2)                   # likelihood ratio
est  = (x_is > theta) * w
p_is, se_is = est.mean(), est.std(ddof=1)/np.sqrt(N5)

print(f"true p        = {true_p:.3e}")
print(f"naive  (N=1e6): {p_naive:.3e} ± {1.96*se_naive:.1e}   "
      f"({int((x_naive>theta).sum())} events seen)")
print(f"IS     (N=1e6): {p_is:.3e} ± {1.96*se_is:.1e}")
c = est[est > 0]                                          # contributions to the estimate
ess_c = c.sum()**2 / (c**2).sum()
print(f"variance-reduction factor ≈ {(se_naive/se_is)**2:,.0f}x")
print(f"naive N needed for IS's accuracy ≈ {N5*(se_naive/se_is)**2:.1e} samples")
print(f"contribution ESS = {ess_c:,.0f} of {len(c):,} contributing samples — healthy")

**A portfolio stress version + the diagnostic that keeps you honest.** P(portfolio loss > 50%) — a ~3.5σ crash for this book, ~2×10⁻⁴: naive MC at N = 400k sees only a few dozen events (relative error ~10%). Tilt the market factor into the crash zone, reweight by the likelihood ratio of the factor alone.

The smoke detector: **effective sample size of the contributions** cᵢ = 1{event}·wᵢ, ESS = (Σc)²/Σc². If a handful of contributions carry the whole estimate (ESS tiny), the CI is fiction regardless of how it looks. (Raw-weight ESS is the conservative version for smooth integrands; for rare-event *indicators* the contribution version is the one that matters — Exercise 4 makes both collapse by overshooting the tilt.)

In [ ]:
def portfolio_loss(zm, zi):                      # one market factor + idiosyncratic
    ret = 0.7*0.20*zm[:, None] + 0.3*0.20*zi     # 10 assets, beta .7 to market
    return -ret.mean(axis=1)                      # loss = -return

N5, d, thresh = 400_000, 10, 0.50
r5 = np.random.default_rng(51)
zm, zi = r5.standard_normal(N5), r5.standard_normal((N5, d))
loss = portfolio_loss(zm, zi)
p_naive = (loss > thresh).mean()
se_nv = np.sqrt(p_naive*(1-p_naive)/N5)

shift = -3.0                                      # tilt the market factor into the crash zone
zm_is = r5.normal(shift, 1.0, size=N5)
w     = np.exp(-shift*zm_is + shift**2/2)         # LR for the market factor only
loss_is = portfolio_loss(zm_is, r5.standard_normal((N5, d)))
est   = (loss_is > thresh) * w
p_is, se_is = est.mean(), est.std(ddof=1)/np.sqrt(N5)
c = est[est > 0]
ess_c = c.sum()**2 / (c**2).sum()

print(f"naive: {p_naive:.2e} ± {1.96*se_nv:.1e} ({int((loss>thresh).sum())} events in {N5:,} draws)")
print(f"IS   : {p_is:.2e} ± {1.96*se_is:.1e}")
print(f"variance-reduction factor ≈ {(se_nv/se_is)**2:,.0f}x")
print(f"contribution ESS = {ess_c:,.0f} of {len(c):,} contributing samples "
      f"— healthy; a tiny ESS would mean the tilt overshot")

## Section 6 — Quasi–Monte Carlo: Sobol' Sequences

Replace iid points with a **low-discrepancy** sequence — deterministic points engineered to fill space evenly (stratification industrialized, in every dimension and every prefix). Convergence ~ (log N)ᵈ/N: close to 1/N, vs MC's 1/√N. **Scrambled** Sobol' (randomized QMC) restores error bars: several independent scramblings = several independent estimates.
> Explained in the guide: [Part 4 - quasi-Monte Carlo](https://statexampro.com/lessons/monte-carlo/variance-reduction/#qmc)


In [ ]:
sob = qmc.Sobol(d=2, scramble=True, rng=np.random.default_rng(60))
pts_sobol = sob.random(512)
pts_iid   = np.random.default_rng(61).uniform(size=(512, 2))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
axes[0].plot(*pts_iid.T, "."); axes[0].set_title("512 iid points (clumps & gaps)")
axes[1].plot(*pts_sobol.T, "."); axes[1].set_title("512 scrambled Sobol' points")
for a in axes: a.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# European call priced by inverse transform: payoff(Phi^{-1}(u)), u from iid vs Sobol
bs = bs_call(S0, K, r, sigma, T)
def price_from_u(u): return euro_payoff(norm.ppf(u)).mean()

Ns = 2 ** np.arange(6, 15)
err_mc, err_q = [], []
for n in Ns:
    e_mc = [abs(price_from_u(np.random.default_rng(1000+i).uniform(size=n)) - bs) for i in range(20)]
    e_q  = [abs(price_from_u(np.clip(qmc.Sobol(1, scramble=True,
             rng=np.random.default_rng(2000+i)).random(n).ravel(), 1e-12, 1-1e-12)) - bs)
            for i in range(20)]
    err_mc.append(np.sqrt(np.mean(np.square(e_mc))))
    err_q.append(np.sqrt(np.mean(np.square(e_q))))

plt.loglog(Ns, err_mc, "o-", label="pseudo-random (MC)")
plt.loglog(Ns, err_q, "s-", label="scrambled Sobol' (QMC)")
plt.loglog(Ns, err_mc[0]*(Ns[0]/Ns)**0.5, "k:", lw=1, label=r"$N^{-1/2}$")
plt.loglog(Ns, err_q[0]*(Ns[0]/Ns)**1.0, "k--", lw=1, label=r"$N^{-1}$")
plt.xlabel("N"); plt.ylabel("RMS pricing error"); plt.legend()
plt.title("QMC vs MC, European call (d=1)"); plt.show()

s_mc = np.polyfit(np.log(Ns), np.log(err_mc), 1)[0]
s_q  = np.polyfit(np.log(Ns), np.log(err_q), 1)[0]
print(f"fitted slopes:  MC {s_mc:.2f} (theory −0.5)   QMC {s_q:.2f} (≈ −1)")

**Does it survive high dimension?** The Asian option is a d = 52 integral. Raw QMC's (log N)ᵈ threat is real, but payoffs like this have low *effective* dimension (a few coordinates carry most of the variance), so Sobol' still wins — and pairing QMC with a Brownian bridge path construction (which concentrates variance in the first coordinates) widens the gap further. Here: plain comparison at d = 52.

In [ ]:
def asian_from_u(u):                        # u shape (n, 52) -> Asian price estimate
    z = norm.ppf(np.clip(u, 1e-12, 1-1e-12))
    dt = T/52
    S = S0*np.exp(np.cumsum((r-.5*sigma**2)*dt + sigma*np.sqrt(dt)*z, axis=1))
    return (np.exp(-r*T)*np.maximum(S.mean(1) - K, 0)).mean()

ref = None   # high-accuracy reference via big CV-run
Yb, Xb = asian_mc(600_000, np.random.default_rng(66))
EXb = geometric_asian_closed_form(S0, K, r, sigma, T, 52)
bcv = np.cov(Yb, Xb)[0,1]/Xb.var(ddof=1)
ref = (Yb - bcv*(Xb - EXb)).mean()

Ns2 = 2 ** np.arange(7, 14)
e_mc, e_q = [], []
for n in Ns2:
    e1 = [abs(asian_from_u(np.random.default_rng(3000+i).uniform(size=(n,52))) - ref) for i in range(12)]
    e2 = [abs(asian_from_u(qmc.Sobol(52, scramble=True,
            rng=np.random.default_rng(4000+i)).random(n)) - ref) for i in range(12)]
    e_mc.append(np.sqrt(np.mean(np.square(e1)))); e_q.append(np.sqrt(np.mean(np.square(e2))))

plt.loglog(Ns2, e_mc, "o-", label="MC"); plt.loglog(Ns2, e_q, "s-", label="Sobol'")
plt.xlabel("N"); plt.ylabel("RMS error"); plt.legend()
plt.title("Asian option, d = 52: QMC still ahead"); plt.show()
print(f"error at N=8192:  MC {e_mc[-1]:.4f}   QMC {e_q[-1]:.4f}   "
      f"advantage {e_mc[-1]/e_q[-1]:.1f}x in error ≈ {(e_mc[-1]/e_q[-1])**2:.0f}x in samples")

## Section 7 — Simulation Studies as a Science: Cluster Trials and Adaptive Designs

**The crime: ignoring clustering.** Cluster-randomized trial — whole clinics assigned to treatment/control; patients within a clinic share a random clinic effect (ICC = 5%). Analyze at the *patient* level with a plain t-test and the false-positive rate explodes, because n_patients overstates the real information. The honest analysis here: t-test on **cluster means**.
> Explained in the guide: [Part 4 - simulation studies as a science](https://statexampro.com/lessons/monte-carlo/variance-reduction/#studies)


In [ ]:
def cluster_trial(k_per_arm, m, icc, effect, rng):
    sd_b, sd_w = np.sqrt(icc), np.sqrt(1 - icc)          # between / within cluster sd
    def arm(shift):
        clin = rng.normal(shift, sd_b, size=(k_per_arm, 1))
        return clin + rng.normal(0, sd_w, size=(k_per_arm, m))
    return arm(0.0), arm(effect)

def analyze(a, b):
    p_bad  = stats.ttest_ind(a.ravel(), b.ravel()).pvalue          # patient-level (WRONG)
    p_good = stats.ttest_ind(a.mean(1), b.mean(1)).pvalue          # cluster means (valid)
    return p_bad < 0.05, p_good < 0.05

r7 = np.random.default_rng(70)
res = np.array([analyze(*cluster_trial(6, 50, 0.05, 0.0, r7)) for _ in range(4000)])
print(f"Type I error, 6 clinics/arm x 50 patients, ICC=0.05, NO true effect:")
print(f"  patient-level t-test : {res[:,0].mean():.1%}   <- catastrophic (nominal 5%)")
print(f"  cluster-means t-test : {res[:,1].mean():.1%}   <- honest")

**Power: patients are cheap, clinics are expensive.** The design effect 1+(m−1)·ICC says information saturates in m; only k (number of clusters) truly buys power. The two power surfaces below are the deliverable for a design meeting — and no closed form covers the messy real versions (unequal cluster sizes, dropout of whole clinics...), while the simulator absorbs them in a line each.

In [ ]:
def power_cluster(k, m, icc=0.05, effect=0.35, n_sim=1200, seed=0):
    r = np.random.default_rng(seed)
    hits = sum(analyze(*cluster_trial(k, m, icc, effect, r))[1] for _ in range(n_sim))
    return hits / n_sim

ks = [4, 6, 8, 10, 14, 18]
for m in [20, 50, 200]:
    pw = [power_cluster(k, m, seed=100+m) for k in ks]
    plt.plot(ks, pw, "o-", label=f"m = {m} patients/clinic")
plt.axhline(0.8, color="k", ls="--")
plt.xlabel("clinics per arm (k)"); plt.ylabel("power"); plt.legend()
plt.title("Cluster-trial power: k buys power, m saturates (design effect)")
plt.show()

**Fixing the peeking crime with a simulation-calibrated boundary.** Part I showed naive interim peeking inflates false positives to ~21%. Group-sequential theory (Pocock, O'Brien–Fleming) fixes this with stricter per-look thresholds — and the thresholds can be *derived by simulation*: simulate the null, record each trial's max |t| across the 5 looks, take the 95th percentile as the critical value. Alpha restored, early stopping kept, no theory required beyond Part I's toolkit.

In [ ]:
def look_stats(rng, max_n=200, step=20):
    a, b = rng.normal(0, 1, max_n), rng.normal(0, 1, max_n)
    return np.array([abs(stats.ttest_ind(a[:n], b[:n]).statistic)
                     for n in range(step, max_n+1, step)])

r7 = np.random.default_rng(71)
null_max = np.array([look_stats(r7).max() for _ in range(4000)])
c_naive, c_cal = 1.96, np.quantile(null_max, 0.95)
print(f"calibrated critical value across 10 looks: {c_cal:.2f}  (vs naive 1.96)")
print(f"false-positive rate: naive {np.mean(null_max > c_naive):.1%} -> calibrated "
      f"{np.mean(null_max > c_cal):.1%}")

# what does the fix cost in power? simulate with a true effect of 0.4 SD
def seq_detect(rng, c, effect, max_n=200, step=20):
    a, b = rng.normal(0, 1, max_n), rng.normal(effect, 1, max_n)
    return any(abs(stats.ttest_ind(a[:n], b[:n]).statistic) > c
               for n in range(step, max_n+1, step))
pw_cal = np.mean([seq_detect(r7, c_cal, 0.4) for _ in range(3000)])
print(f"power at effect=0.4 with the calibrated sequential design: {pw_cal:.1%}")

**How many simulation replications does a simulation study need?** A power estimate is a proportion: SE = √(p(1−p)/n_sim). Targeting ±1% at p≈0.8 → n_sim ≈ 6,400; ±0.5% → 25,600. Report this like any other error bar — a simulation study is an experiment and gets the full experimental-design treatment (in the literature: the ADEMP framework — Aims, Data-generating mechanisms, Estimands, Methods, Performance measures).

In [ ]:
for half_width in [0.02, 0.01, 0.005]:
    n_sim = 0.8*0.2*(1.96/half_width)**2
    print(f"±{half_width:.1%} on a power ≈ 0.8  ->  n_sim ≈ {n_sim:,.0f}")

## Section 8 — The Bridge to MCMC

When you can only *evaluate* an unnormalized density (every Bayesian posterior), direct sampling dies. **Random-walk Metropolis**: propose a nearby move; accept with probability min(1, p(new)/p(old)); on rejection, *repeat* the current point. The chain's long-run distribution is exactly p — but samples are **correlated**, so N chain draws ≠ N iid draws. The exchange rate is the **effective sample size**: N_eff = N / (1 + 2Σρₖ).

Test problem with a known answer (certification, as always): posterior of a coin's bias after 7 heads in 10 tosses, uniform prior → Beta(8, 4).
> Explained in the guide: [Part 4 - the bridge to MCMC](https://statexampro.com/lessons/monte-carlo/variance-reduction/#mcmc)


In [ ]:
def log_post(p):     # unnormalized: 7 heads, 3 tails, uniform prior
    return 7*np.log(p) + 3*np.log(1-p) if 0 < p < 1 else -np.inf

def metropolis(n, step, seed):
    r = np.random.default_rng(seed)
    x, lp = 0.5, log_post(0.5)
    out, acc = np.empty(n), 0
    for i in range(n):
        prop = x + r.normal(0, step)
        lpp = log_post(prop)
        if np.log(r.uniform()) < lpp - lp:
            x, lp, acc = prop, lpp, acc + 1
        out[i] = x
    return out, acc / n

chain, acc_rate = metropolis(60_000, step=0.30, seed=80)
burn = chain[2000:]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(chain[:1500], lw=0.7); axes[0].set_title(f"trace (acceptance {acc_rate:.0%})")
axes[1].hist(burn, bins=80, density=True, alpha=0.6, label="Metropolis")
g = np.linspace(0.2, 1, 300)
axes[1].plot(g, stats.beta.pdf(g, 8, 4), "k", lw=2, label="true Beta(8,4)")
axes[1].legend(); axes[1].set_title("chain vs truth")
plt.tight_layout(); plt.show()

In [ ]:
def ess(x, max_lag=200):
    x = x - x.mean()
    acf = np.array([1.0] + [np.corrcoef(x[:-k], x[k:])[0, 1] for k in range(1, max_lag)])
    pos = np.argmax(acf < 0) if (acf < 0).any() else max_lag       # truncate at first negative
    return len(x) / (1 + 2 * acf[1:pos].sum()), acf

n_eff, acf = ess(burn)
print(f"chain length {len(burn):,} -> effective sample size ≈ {n_eff:,.0f} "
      f"({n_eff/len(burn):.0%} efficiency)")
print(f"posterior mean: chain {burn.mean():.4f} ± {1.96*burn.std()/np.sqrt(n_eff):.4f} "
      f"(SE uses N_eff!)   truth {8/12:.4f}")

plt.plot(acf[:80], "o-", ms=3); plt.xlabel("lag"); plt.ylabel("autocorrelation")
plt.title("The price of Metropolis: correlated draws"); plt.show()

Step-size is a real dial: too small → glacial diffusion (high acceptance, huge autocorrelation); too big → constant rejection (chain freezes in place). ~20–50% acceptance is the folk optimum for random-walk proposals. Modern samplers (HMC/NUTS in Stan, PyMC) automate all of this — but the diagnostics you just built (trace, acceptance, autocorrelation, N_eff, multi-chain R̂) remain exactly what professionals stare at.

## Exercises

1. **(Drift check.)** Under the risk-neutral measure the *discounted* price e^{−rt}S_t must be a martingale: E[e^{−rT}S_T] = S₀. Verify to within an error bar; then break it with the −σ²/2 bug and watch the martingale property fail.
2. **(Put–call parity as a free control.)** Price the put with the call+parity as a control variate (E[call−put] = S₀ − Ke^{−rT}, exact). What reduction factor do you get vs naive put pricing?
3. **(Copula stress.)** Re-run the Section 3 portfolio with a t-copula (df = 3) instead of Gaussian at ρ = 0.3. How much does P5 worsen with *identical* marginals and ρ?
4. **(IS overshoot.)** In Section 5's tail example, push the shift θ to 8 and watch ESS collapse and the CI go quietly wrong. Plot estimate ± CI vs θ ∈ [2, 8] against the true value.
5. **(Sequential design.)** Calibrate a *one-sided* 3-look boundary (looks at n = 60, 120, 200) for α = 2.5% and estimate its power at effect 0.35 and its expected sample size (sum over looks of stopping probabilities). Compare with the fixed-n design of equal power.

Solution sketches below.

In [ ]:
r9 = np.random.default_rng(99)

# 1) martingale check
Z = r9.standard_normal(1_000_000)
ST_ok  = S0*np.exp((r-.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
ST_bug = S0*np.exp(r*T + sigma*np.sqrt(T)*Z)
for name, s in [("correct", ST_ok), ("buggy", ST_bug)]:
    d = np.exp(-r*T)*s
    print(f"1) {name:7s}: E[e^-rT S_T] = {d.mean():.3f} ± {1.96*d.std()/1000:.3f}  (S0 = {S0})")

# 2) put with parity control
put  = np.exp(-r*T)*np.maximum(K - ST_ok, 0)
call = np.exp(-r*T)*np.maximum(ST_ok - K, 0)
X = call - put; EX = S0 - K*np.exp(-r*T)
b = np.cov(put, X)[0,1]/X.var(ddof=1)
put_cv = put - b*(X - EX)
print(f"\n2) reduction factor = {put.var()/put_cv.var():,.0f}x   "
      f"put = {put_cv.mean():.4f} (BS parity: {bs_call(S0,K,r,sigma,T) - EX:.4f})")

# 4) IS overshoot: ESS collapse
theta_true = norm.sf(4.5)
for th in [3, 4.5, 6, 8]:
    x = r9.normal(th, 1.0, 300_000)
    w = np.exp(-th*x + th**2/2)
    est = (x > 4.5)*w
    c = est[est > 0]
    e = c.sum()**2/(c**2).sum()
    print(f"4) shift {th:.1f}: est {est.mean():.2e} ± {1.96*est.std()/np.sqrt(len(est)):.0e}"
          f"   contribution ESS {e:9,.0f}   (true {theta_true:.2e})")

---
**That's the toolkit.** GBM and risk-neutral pricing certified against Black–Scholes; CRN Greeks; Cholesky and copulas; antithetic / control variates / stratification with measured reduction factors; importance sampling with ESS diagnostics; scrambled Sobol'; cluster-trial power and simulation-calibrated sequential boundaries; Metropolis with N_eff. The guide's Chapter 10 assembles it into a pre-flight checklist for production simulations.

---

### Back to the guide

[The four parts](https://statexampro.com/lessons/monte-carlo/) &middot; [Part 1](https://statexampro.com/lessons/monte-carlo/first-principles/) &middot; [Part 2](https://statexampro.com/lessons/monte-carlo/recipe-and-projects/) &middot; [Part 3](https://statexampro.com/lessons/monte-carlo/paths-and-options/) &middot; [Part 4](https://statexampro.com/lessons/monte-carlo/variance-reduction/)

The guide and these notebooks are free, and stay that way because of the app: **Stat Exam Pro** drills the
exam-style questions behind these ideas - the traps, with worked answers, in English and French.
[Get it on the App Store](https://apps.apple.com/us/app/stat-exam-pro/id6755912771).
